# B cell preprocessing

Download the source IGVF B-cell data from https://data.igvf.org before running this notebook. The expected raw file for this reproduction path is `IGVFFI3928IUMP.h5ad`, available under `data/flowmap_manuscript/b_cell` locally. The notebook writes `bcell_velocity_standard.h5ad`, which is used by the embedding and curvature notebooks.


In [ ]:
import scanpy as sc
import scvelo as scv
import anndata as ad
import numpy as np

# --------------------------
# 0. Settings (REPRODUCIBILITY)
# --------------------------

from pathlib import Path

cwd = Path.cwd()
ANALYSIS_DIR = (
    cwd
    if cwd.name == "06_b_cell_igvf"
    else Path("06_b_cell_igvf")
    if Path("06_b_cell_igvf").exists()
    else Path("..").resolve()
    if cwd.name == "notebooks"
    else Path("../..").resolve()
)
DATA_DIR = Path("data/flowmap_manuscript/b_cell")
if not DATA_DIR.exists():
    DATA_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell"
RESULTS_DIR = Path("data/flowmap_manuscript/b_cell_results")
if not RESULTS_DIR.exists():
    RESULTS_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell_results"
FIGURE_DIR = ANALYSIS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
UTILS_DIR = ANALYSIS_DIR / "utils"

sc.settings.verbosity = 3
scv.settings.verbosity = 3

np.random.seed(0)

In [ ]:
# --------------------------
# 1. Load
# --------------------------
adata = ad.read_h5ad(DATA_DIR / "IGVFFI3928IUMP.h5ad")
adata

In [ ]:
# --------------------------
# 2. Fast QC (NO calculate_qc)
# --------------------------

# basic filters only (fast)
sc.pp.filter_cells(adata, min_counts=1000)
sc.pp.filter_cells(adata, min_genes=500)

sc.pp.filter_genes(adata, min_cells=20)

# optional: very cheap mito filter (vectorized, no heavy metrics)
mt_mask = adata.var_names.str.startswith("MT-")
pct_mt = (
    np.array(adata[:, mt_mask].X.sum(axis=1)).flatten() /
    np.array(adata.X.sum(axis=1)).flatten()
)

adata = adata[pct_mt < 0.2].copy()
adata

In [ ]:
# --------------------------
# 3. Set layers (CRITICAL for scVelo)
# --------------------------
adata.layers["spliced"] = adata.layers["mature"]
adata.layers["unspliced"] = adata.layers["nascent"]

# --------------------------
# 4. scVelo preprocessing (DO NOT use scanpy normalize here)
# --------------------------
scv.pp.filter_and_normalize(
    adata,
    min_shared_counts=20,
    n_top_genes=2000
)

In [ ]:
# --------------------------
# 5. Moments (graph construction)
# --------------------------
scv.pp.moments(
    adata,
    n_pcs=30,
    n_neighbors=30
)

In [ ]:
scv.tl.velocity(adata, mode="stochastic")
scv.tl.velocity_graph(adata)

In [ ]:
sc.tl.umap(adata, random_state=0)
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap"
)

In [ ]:
import mygene
import pandas as pd
import numpy as np

def clean_gene_names(adata):
    mg = mygene.MyGeneInfo()
    
    # 1. Prepare query list (clean versioning like .1, .2)
    original_ids = adata.var_names.tolist()
    clean_ids = [str(g).split('.')[0] for g in original_ids]
    
    # 2. Query as a list of dicts (more robust than the dataframe output)
    print(f"Querying {len(clean_ids)} genes...")
    results = mg.querymany(
        clean_ids, 
        scopes="ensembl.gene", 
        fields="symbol", 
        species="human", 
        as_dataframe=False, # Use list of dicts to avoid index/KeyErrors
        verbose=False
    )
    
    # 3. Build a strict mapping: Only add if 'symbol' actually exists
    symbol_map = {}
    for item in results:
        query_id = item.get('query')
        symbol = item.get('symbol')
        if query_id and symbol and str(symbol).lower() != 'nan':
            symbol_map[query_id] = str(symbol)

    # 4. Final assignment: Map it or Keep it
    new_names = []
    for i, orig in enumerate(original_ids):
        clean = clean_ids[i]
        # Priority: 1. Found Symbol, 2. Cleaned ID, 3. Original string
        final_name = symbol_map.get(clean, clean if str(clean).lower() != 'nan' else orig)
        new_names.append(final_name)
    
    # 5. Update AnnData
    adata.var['original_id'] = original_ids # Keep a backup just in case
    adata.var_names = new_names
    adata.var_names_make_unique()
    
    print(f"Mapping complete. Unique names assigned.")
    return adata

# Execution
adata = clean_gene_names(adata)
adata

In [ ]:
scv.tl.recover_dynamics(adata)
scv.tl.latent_time(adata)

In [ ]:
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="velocity_pseudotime"
)
adata.write(DATA_DIR / "bcell_velocity_standard.h5ad")